# SASMaker tutorial

In this tutorial, you will:

1. Model a small electrical substation.
2. Add three intelligent electronic devices (IEDs).
3. Simulate a protection event.
4. Generate IEC 61850 GOOSE traffic.
5. inspect the traffic in Wireshark.

Run each cell in order using the **Run** button.

In [6]:
from pathlib import Path

from sasmaker import Substation
from sasmaker.builder import snap_child_to_slot
from sasmaker.plotting import plot_one_line
from sasmaker.simulation import (
    Simulation,
    sample_ieds,
    trigger_ied_cb_trip,
)
from sasmaker.util import (
    capture_to_pcap,
    create_interfaces,
    generate_values_df,
    spawn_script,
)

## Create substation A1

A1 is a small 20 kV substation with:

- One incoming line
- Two outgoing feeders
- Three circuit breakers
- Three current transformers
- Three IEDs

IED1 monitors the incoming line. IED2 and IED3 monitor the two outgoing feeders.

In [ ]:
substation = Substation("A1 workshop substation")

upstream_bus = substation.add_busbar(
    "Upstream network",
    vn_kv=20,
    x=1.0,
    y=1.0,
    draw_slots=1,
    ext_grid=True,
)

main_bus = substation.add_busbar(
    "20 kV busbar",
    vn_kv=20,
    x=1.0,
    y=0.0,
    draw_slots=3,
)

snap_child_to_slot(
    substation,
    upstream_bus,
    main_bus,
    slot_idx=0,
    drop=0.4,
)

substation.add_ext_grid("External grid", upstream_bus)

line1 = substation.add_line(
    "L1",
    upstream_bus,
    main_bus,
    length_km=0.5,
)

cb1 = substation.add_cb("CB-1", line1, side="to")
ct1 = substation.add_ct("CT-1", line1, side="to")
ied1 = substation.add_ied("IED1", ct=ct1, cb=cb1)

ied1.ptoc.pickup_ka = 20.0

feeder_bus1 = substation.add_busbar(
    "Feeder 1 bus",
    vn_kv=20,
    x=1.0,
    y=-1.0,
    draw_slots=1,
    draw_label=False,
)

snap_child_to_slot(
    substation,
    main_bus,
    feeder_bus1,
    slot_idx=0,
)

line2 = substation.add_line(
    "L2",
    main_bus,
    feeder_bus1,
    length_km=0.5,
)

cb2 = substation.add_cb("CB-2", line2, side="from")
ct2 = substation.add_ct("CT-2", line2, side="from")
ied2 = substation.add_ied("IED2", ct=ct2, cb=cb2)

substation.add_load(
    "Load 1",
    feeder_bus1,
    p_mw=15,
    q_mvar=10,
)

feeder_bus2 = substation.add_busbar(
    "Feeder 2 bus",
    vn_kv=20,
    x=1.0,
    y=-1.0,
    draw_slots=1,
    draw_label=False,
)

snap_child_to_slot(
    substation,
    main_bus,
    feeder_bus2,
    slot_idx=2,
)

line3 = substation.add_line(
    "L3",
    main_bus,
    feeder_bus2,
    length_km=0.5,
)

cb3 = substation.add_cb("CB-3", line3, side="from")
ct3 = substation.add_ct("CT-3", line3, side="from")
ied3 = substation.add_ied("IED3", ct=ct3, cb=cb3)

substation.add_load(
    "Load 2",
    feeder_bus2,
    p_mw=15,
    q_mvar=10,
)

In [ ]:
# Cell run once after building the substation
from copy import deepcopy
substation_template = deepcopy(substation)

plot_one_line(
    substation,
    line_color="#009B24",
    label_buses=True,
    label_lines=False,
)

## Simulate a protection event

The simulation runs for 20 seconds. At 10 seconds, IED2 issues a trip command. Its circuit breaker then opens and disconnects feeder 1.

In [ ]:
simulation_length = 20

substation = deepcopy(substation_template)
ieds = list(substation.ieds.values())

simulation = Simulation(
    "IED2 protection trip",
    t_end=simulation_length,
    dt=1.0,
)
simulation.add_sampler(sample_ieds())
trigger_ied_cb_trip(simulation, ied2, t0=10)

simulation_data = simulation.run(substation)
simulation_data[[
    "t",
    "ied:IED2:protection_tripped",
    "cb:CB-2:closed",
]]

## Prepare the GOOSE publisher values

SASMaker converts the electrical simulation into one `value.csv` file for each IED. These files are the inputs to the existing libiec61850-based publishers.

In [ ]:
working_directory = Path.cwd()
if working_directory.name == "workshop":
    source_directory = working_directory.parent
elif (working_directory / "src" / "toolchain").is_dir():
    source_directory = working_directory / "src"
else:
    source_directory = working_directory

toolchain_directory = source_directory / "toolchain"
ieds = list(substation.ieds.values())

for ied in ieds:
    values = generate_values_df(simulation_data, ied)
    output_file = toolchain_directory / ied.name / "value.csv"
    values.to_csv(output_file, header=False, index=False)
    print(f"Created {output_file}")

## Export the GOOSE traffic to a PCAP

The next cell starts the existing C publishers, captures their real Ethernet traffic, and then removes the temporary interfaces. The output can be opened in Wireshark with the display filter `goose`.

In [ ]:
capture_file = source_directory / "workshop" / "output" / "sesbc_workshop.pcap"

if capture_file.exists():
    capture_file.unlink()

capture_path = capture_to_pcap(
    ieds=ieds,
    duration=simulation_length,
    output=capture_file,
    toolchain_directory=toolchain_directory,
)

print(f"Capture saved to {capture_path}")

In [ ]:
import importlib
import helpers.pcap_parser

importlib.reload(helpers.pcap_parser)

print(helpers.pcap_parser.__file__)

packet_data = helpers.pcap_parser.parse_goose_pcap(capture_path)

print(f"Parsed {len(packet_data)} GOOSE packets")
packet_data.head(n=20)

In [ ]:
import helpers.plotting

importlib.reload(helpers.plotting)

helpers.plotting.plot_goose_overview(
    packet_data,
    bin_width=0.1,
);